# 🌐 Análisis Exploratorio del Proyecto Nacional de Fibra Óptica de Colombia

---

**Fecha:** Marzo 2025  
**Fuente de datos:** Proyecto Nacional de Fibra Óptica - MinTIC Colombia  

## 📋 Descripción del Notebook

Este notebook realiza un análisis exploratorio exhaustivo del **Proyecto Nacional de Fibra Óptica** de Colombia,
el cual tiene como objetivo llevar conectividad de alta velocidad a municipios de todo el territorio nacional.

### Objetivos del análisis:
- Explorar la distribución geográfica de los municipios beneficiados
- Analizar las inversiones por región, departamento y municipio
- Estudiar la evolución temporal de los despliegues
- Identificar patrones y correlaciones en los datos
- Generar visualizaciones informativas para la toma de decisiones

### Contenido:
1. Carga y limpieza de datos
2. Estadísticas descriptivas
3. Distribución geográfica
4. Análisis temporal
5. Análisis de inversión
6. Correlaciones y relaciones
7. Hallazgos y recomendaciones


## 📦 Instalación de Paquetes Requeridos

Instalamos las bibliotecas necesarias para el análisis de datos y visualización.

In [ ]:
!pip install pandas numpy matplotlib seaborn plotly folium chardet geopandas -q
print("✅ Paquetes instalados correctamente")

In [ ]:
# ============================================================
# IMPORTACIÓN DE BIBLIOTECAS
# ============================================================

# Manejo de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Geográficas
import folium
from folium.plugins import MarkerCluster

# Utilidades
import chardet
import warnings
from datetime import datetime

# Configuración
warnings.filterwarnings('ignore')

# Configuración de estilo para matplotlib
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 13
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11
plt.rcParams['figure.dpi'] = 100

# Configuración de Seaborn
sns.set_theme(style="whitegrid", palette="husl", font_scale=1.1)

# Configuración de Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.width', 200)

# Paleta de colores profesional
COLORES_REGIONES = {
    'Región Caribe': '#FF6B6B',
    'Región Centro Oriente': '#4ECDC4',
    'Región Eje Cafetero': '#45B7D1',
    'Región Pacífico': '#96CEB4',
    'Región Centro Sur': '#FFEAA7',
    'Región Llano': '#DDA0DD'
}

PALETA_COLORES = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD',
                  '#98D8C8', '#F7DC6F', '#BB8FCE', '#85C1E9', '#F0B27A', '#82E0AA']

print("✅ Bibliotecas importadas y configuradas correctamente")
print(f"📋 Versiones: pandas={pd.__version__}, numpy={np.__version__}")

## 📂 Carga de Datos y Corrección de Codificación

Cargamos el dataset del Proyecto Nacional de Fibra Óptica y aplicamos correcciones
de codificación para manejar caracteres especiales del español (ó, í, ñ, etc.).

In [ ]:
# ============================================================
# CARGA DE DATOS Y CORRECCIÓN DE CODIFICACIÓN
# ============================================================

# Ruta del archivo CSV
ruta_archivo = '/content/Proyecto_Nacional_de_Fibra_Óptica_20260314.csv'

# Detección de codificación
with open(ruta_archivo, 'rb') as f:
    resultado = chardet.detect(f.read())
codificacion_detectada = resultado['encoding']
confianza = resultado['confidence']
print(f"🔍 Codificación detectada: {codificacion_detectada} (confianza: {confianza:.2%})")

# Intentar lectura con diferentes codificaciones
for enc in ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']:
    try:
        df = pd.read_csv(ruta_archivo, encoding=enc)
        print(f"✅ Archivo leído exitosamente con codificación: {enc}")
        break
    except Exception as e:
        print(f"❌ Error con codificación {enc}: {e}")

# Función para corregir caracteres garbled
def corregir_caracteres(texto):
    """Corrige caracteres especiales garbled del español."""
    if not isinstance(texto, str):
        return texto
    reemplazos = {
        'ó': 'ó', 'Ó': 'Ó',
        'í': 'í', 'Í': 'Í',
        'á': 'á', 'Á': 'Á',
        'é': 'é', 'É': 'É',
        'ú': 'ú', 'Ú': 'Ú',
        'ñ': 'ñ', 'Ñ': 'Ñ',
        'ü': 'ü', 'Ü': 'Ü',
    }
    for viejo, nuevo in reemplazos.items():
        texto = texto.replace(viejo, nuevo)
    return texto

# Aplicar corrección a todas las columnas de texto
for col in df.columns:
    df[col] = df[col].apply(lambda x: corregir_caracteres(str(x)) if isinstance(x, str) else x)
    if df[col].dtype == 'object':
        df[col] = df[col].str.strip()

print(f"\n📊 Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"\n{'='*80}")
print("VISTA PREVIA DE LOS DATOS")
print(f"{'='*80}")
df.head(10)

## 📝 Descripción de las Columnas del Dataset

| Columna | Descripción |
|---------|-------------|
| **REGION** | Región geográfica de Colombia a la que pertenece el municipio (6 regiones) |
| **DEPARTAME_COD** | Código numérico del departamento |
| **DEPARTAME_NOMBRE** | Nombre del departamento (27 departamentos) |
| **MUNICIPIO_NOMBRE** | Nombre del municipio beneficiado |
| **FECHA OPERACION** | Fecha de inicio de operación del servicio de fibra óptica |
| **FECHA FIN OPERACION** | Fecha de finalización estimada de la operación |
| **ESTADO ACTUAL** | Estado actual del proyecto (todos en "En Operación") |
| **MUNICIPIO_COD** | Código único del municipio según DANE |
| **FEC CARGUE** | Fecha de carga del registro al sistema |
| **ID_INDICADOR** | Identificador del indicador de seguimiento |
| **INVERSION FONTIC** | Inversión del Fondo Único de TIC (FONTIC) en COP |
| **INVERSION CONTRAPARTIDA** | Inversión de contrapartida (departamental/municipal) en COP |
| **INVERSION TOTAL** | Inversión total del proyecto (FONTIC + Contrapartida) en COP |
| **FECHA VIGENCIA** | Fecha de vigencia del registro |

## 🔍 Exploración Básica de los Datos

Realizamos una exploración inicial para entender la estructura y calidad del dataset.

In [ ]:
# ============================================================
# EXPLORACIÓN BÁSICA
# ============================================================

print("=" * 80)
print("INFORMACIÓN GENERAL DEL DATASET")
print("=" * 80)

print(f"\n📐 Dimensiones: {df.shape[0]} filas × {df.shape[1]} columnas")

print(f"\n{'='*80}")
print("TIPOS DE DATOS POR COLUMNA")
print(f"{'='*80}")
print(df.dtypes)

print(f"\n{'='*80}")
print("VALORES NULOS POR COLUMNA")
print(f"{'='*80}")
nulos = df.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else "✅ No se encontraron valores nulos")

print(f"\n{'='*80}")
print("VALORES ÚNICOS POR COLUMNA")
print(f"{'='*80}")
for col in df.columns:
    n_unicos = df[col].nunique()
    print(f"  {col:30s}: {n_unicos:5d} valores únicos")

print(f"\n{'='*80}")
print("PRIMERAS 5 FILAS")
print(f"{'='*80}")
display(df.head())

print(f"\n{'='*80}")
print("ÚLTIMAS 5 FILAS")
print(f"{'='*80}")
display(df.tail())

print(f"\n{'='*80}")
print("DISTRIBUCIÓN POR REGIÓN")
print(f"{'='*80}")
print(df['REGION'].value_counts())

print(f"\n{'='*80}")
print("TOP 10 DEPARTAMENTOS POR CANTIDAD DE MUNICIPIOS")
print(f"{'='*80}")
print(df['DEPARTAME_NOMBRE'].value_counts().head(10))

## 🧹 Limpieza y Transformación de Datos

Realizamos la limpieza de datos, conversión de tipos y creación de columnas derivadas
para facilitar el análisis posterior.

In [ ]:
# ============================================================
# LIMPIEZA Y TRANSFORMACIÓN DE DATOS
# ============================================================

# 1. Limpiar espacios en blanco en columnas de texto
columnas_texto = df.select_dtypes(include='object').columns
for col in columnas_texto:
    df[col] = df[col].astype(str).str.strip()

# Corrección de nombres de regiones con caracteres garbled para que coincidan con la paleta de colores
mapping_regions = {
    'Regi�n Caribe': 'Región Caribe',
    'Regi�n Centro Orient': 'Región Centro Oriente',
    'Regi�n Eje Cafetero': 'Región Eje Cafetero',
    'Regi�n Pac�fico': 'Región Pacífico',
    'Regi�n Centro Sur': 'Región Centro Sur',
    'Regi�n Llano': 'Región Llano'
}
df['REGION'] = df['REGION'].replace(mapping_regions)
print("✅ Nombres de regiones corregidos.")

# 2. Convertir columnas de fecha a datetime
columnas_fecha = ['FECHA OPERACION', 'FECHA FIN OPERACION', 'FEC CARGUE', 'FECHA VIGENCIA']
for col in columnas_fecha:
    df[col] = pd.to_datetime(df[col], errors='coerce')
    print(f"📅 {col}: convertido a datetime")

# 3. Convertir columnas de inversión a numérico
columnas_inversion = ['INVERSION FONTIC', 'INVERSION CONTRAPARTIDA', 'INVERSION TOTAL']
for col in columnas_inversion:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    print(f"💰 {col}: convertido a numérico")

# 4. Crear columnas derivadas

# Año de inicio de operación
df['AÑO_INICIO'] = df['FECHA OPERACION'].dt.year
print("\n🆕 Columna 'AÑO_INICIO' creada")

# Mes-Año de inicio
df['MES_AÑO_INICIO'] = df['FECHA OPERACION'].dt.to_period('M')
print("🆕 Columna 'MES_AÑO_INICIO' creada")

# Duración en años (diferencia entre fecha fin y fecha inicio)
df['DURACION_DIAS'] = (df['FECHA FIN OPERACION'] - df['FECHA OPERACION']).dt.days
df['DURACION_AÑOS'] = df['DURACION_DIAS'] / 365.25
print("🆕 Columnas 'DURACION_DIAS' y 'DURACION_AÑOS' creadas")

# Inversión total en billones de COP
df['INVERSION_TOTAL_BILLONES'] = df['INVERSION TOTAL'] / 1e9
print("🆕 Columna 'INVERSION_TOTAL_BILLONES' creada")

# Inversión FONTIC en billones de COP
df['INVERSION_FONTIC_BILLONES'] = df['INVERSION FONTIC'] / 1e9
print("🆕 Columna 'INVERSION_FONTIC_BILLONES' creada")

# Inversión Contrapartida en billones de COP
df['INVERSION_CONTRAPARTIDA_BILLONES'] = df['INVERSION CONTRAPARTIDA'] / 1e9
print("🆕 Columna 'INVERSION_CONTRAPARTIDA_BILLONES' creada")

# Porcentaje de contrapartida sobre el total
df['PORCENTAJE_CONTRAPARTIDA'] = (df['INVERSION CONTRAPARTIDA'] / df['INVERSION TOTAL']) * 100
df['PORCENTAJE_FONTIC'] = (df['INVERSION FONTIC'] / df['INVERSION TOTAL']) * 100
print("🆕 Columnas 'PORCENTAJE_CONTRAPARTIDA' y 'PORCENTAJE_FONTIC' creadas")

print(f"\n{'='*80}")
print("RESUMEN DEL DATASET LIMPIO")
print(f"{'='*80}")
print(df.info())

print(f"\n{'='*80}")
print("ESTADÍSTICAS BÁSICAS DE COLUMNAS NUMÉRICAS")
print(f"{'='*80}")
display(df.describe())

---
# 📊 1. Estadísticas Descriptivas

En esta sección analizamos las estadísticas descriptivas del dataset para entender
las características principales de las variables numéricas y categóricas.

In [ ]:
# ============================================================
# ESTADÍSTICAS DESCRIPTIVAS DETALLADAS
# ============================================================

print("=" * 80)
print("ESTADÍSTICAS DESCRIPTIVAS - VARIABLES NUMÉRICAS")
print("=" * 80)

# Selección de columnas numéricas relevantes
columnas_num = ['INVERSION FONTIC', 'INVERSION CONTRAPARTIDA', 'INVERSION TOTAL',
                'DURACION_AÑOS', 'PORCENTAJE_FONTIC', 'PORCENTAJE_CONTRAPARTIDA']

# Tabla descriptiva personalizada
stats_df = df[columnas_num].describe().T
stats_df['mediana'] = df[columnas_num].median()
stats_df['rango'] = stats_df['max'] - stats_df['min']
stats_df['cv'] = (stats_df['std'] / stats_df['mean'] * 100).round(2)
stats_df = stats_df[['count', 'mean', 'std', 'min', '25%', 'mediana', '75%', 'max', 'rango', 'cv']]
stats_df.columns = ['N', 'Media', 'Desv. Est.', 'Mínimo', 'Q1', 'Mediana', 'Q3', 'Máximo', 'Rango', 'CV(%)']

# Formatear para mejor legibilidad
for col in ['Media', 'Desv. Est.', 'Mínimo', 'Q1', 'Mediana', 'Q3', 'Máximo', 'Rango']:
    stats_df[col] = stats_df[col].apply(lambda x: f"${x:,.0f}" if abs(x) > 100 else f"{x:.2f}")

display(stats_df)

print(f"\n{'='*80}")
print("ESTADÍSTICAS DESCRIPTIVAS - VARIABLES CATEGÓRICAS")
print(f"{'='*80}")

# Variables categóricas
print(f"\n📌 Regiones ({df['REGION'].nunique()}):")
for region, count in df['REGION'].value_counts().items():
    pct = count / len(df) * 100
    print(f"   • {region}: {count} municipios ({pct:.1f}%)")

print(f"\n📌 Departamentos ({df['DEPARTAME_NOMBRE'].nunique()}):")
for dept, count in df['DEPARTAME_NOMBRE'].value_counts().head(10).items():
    pct = count / len(df) * 100
    print(f"   • {dept}: {count} municipios ({pct:.1f}%)")

print(f"\n📌 Años de inicio de operación:")
for anio, count in df['AÑO_INICIO'].value_counts().sort_index().items():
    pct = count / len(df) * 100
    print(f"   • {int(anio)}: {count} municipios ({pct:.1f}%)")

print(f"\n{'='*80}")
print("RESUMEN DE INVERSIÓN TOTAL")
print(f"{'='*80}")
print(f"   💰 Inversión total acumulada: ${df['INVERSION TOTAL'].sum():,.0f} COP")
print(f"   💰 Inversión total acumulada: ${df['INVERSION TOTAL'].sum()/1e12:,.2f} billones COP")
print(f"   💰 Inversión promedio por municipio: ${df['INVERSION TOTAL'].mean():,.0f} COP")
print(f"   💰 Inversión promedio por municipio: ${df['INVERSION TOTAL'].mean()/1e9:,.2f} mil millones COP")
print(f"   💰 Inversión FONTIC total: ${df['INVERSION FONTIC'].sum():,.0f} COP")
print(f"   💰 Inversión Contrapartida total: ${df['INVERSION CONTRAPARTIDA'].sum():,.0f} COP")
print(f"   📊 % promedio FONTIC: {df['PORCENTAJE_FONTIC'].mean():.1f}%")
print(f"   📊 % promedio Contrapartida: {df['PORCENTAJE_CONTRAPARTIDA'].mean():.1f}%")
print(f"   ⏱️  Duración promedio de operación: {df['DURACION_AÑOS'].mean():.1f} años")

---
# 🗺️ 2. Distribución Geográfica

Analizamos cómo se distribuyen los municipios beneficiados por el proyecto a nivel
de región y departamento, identificando las áreas con mayor y menor cobertura.

In [ ]:
# ============================================================
# GRÁFICO 1: Municipios por Región (Barras Horizontales)
# ============================================================

# Contar municipios por región
municipios_region = df['REGION'].value_counts().sort_values(ascending=True)

# Crear figura
fig, ax = plt.subplots(figsize=(14, 8))

# Colores por región
colores = [COLORES_REGIONES.get(r, '#999999') for r in municipios_region.index]

# Gráfico de barras horizontales
bars = ax.barh(municipios_region.index, municipios_region.values, color=colores,
               edgecolor='white', linewidth=1.5, height=0.65)

# Etiquetas de valor en las barras
for bar, valor in zip(bars, municipios_region.values):
    pct = valor / len(df) * 100
    ax.text(bar.get_width() + 3, bar.get_y() + bar.get_height()/2,
            f'{valor} ({pct:.1f}%)', va='center', ha='left', fontsize=12, fontweight='bold')

# Configuración
ax.set_xlabel('Número de Municipios', fontsize=13, fontweight='bold')
ax.set_ylabel('')
ax.set_title('Número de Municipios con Fibra Óptica por Región',
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlim(0, max(municipios_region.values) * 1.25)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='y', labelsize=12)

# Anotación con el total
ax.text(0.98, 0.02, f'Total: {len(df)} municipios en {df["REGION"].nunique()} regiones',
        transform=ax.transAxes, fontsize=11, ha='right', va='bottom',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='lightblue', alpha=0.8))

plt.tight_layout()
plt.savefig('grafico_01_municipios_por_region.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: grafico_01_municipios_por_region.png")

In [ ]:
# ============================================================
# GRÁFICO 2: Top 15 Departamentos por Municipios + Otros
# ============================================================

# Contar municipios por departamento
dept_counts = df['DEPARTAME_NOMBRE'].value_counts()

# Top 15 + Otros
top_15 = dept_counts.head(15)
otros = pd.Series({'OTROS': dept_counts.iloc[15:].sum()}, name='count')
dept_plot = pd.concat([top_15, otros]).sort_values(ascending=True)

# Mapear región a cada departamento para colorear
dept_region = df.groupby('DEPARTAME_NOMBRE')['REGION'].first()
dept_plot_region = dept_plot.index.map(lambda x: dept_region.get(x, 'Otro'))
colores_dept = [COLORES_REGIONES.get(r, '#BBBBBB') for r in dept_plot_region]

# Crear figura
fig, ax = plt.subplots(figsize=(14, 10))

bars = ax.barh(dept_plot.index, dept_plot.values, color=colores_dept,
               edgecolor='white', linewidth=1.0, height=0.7)

# Etiquetas de valor
for bar, valor in zip(bars, dept_plot.values):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            str(valor), va='center', ha='left', fontsize=10, fontweight='bold')

# Configuración
ax.set_xlabel('Número de Municipios', fontsize=13, fontweight='bold')
ax.set_title('Top 15 Departamentos con Más Municipios Beneficiados por Fibra Óptica',
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlim(0, max(dept_plot.values) * 1.15)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='y', labelsize=10)

# Leyenda de regiones
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=r) for r, c in COLORES_REGIONES.items()]
legend_elements.append(Patch(facecolor='#BBBBBB', label='Otros'))
ax.legend(handles=legend_elements, loc='lower right', fontsize=9, title='Región', title_fontsize=10)

plt.tight_layout()
plt.savefig('grafico_02_top_departamentos.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: grafico_02_top_departamentos.png")

In [ ]:
# ============================================================
# GRÁFICO 3: Gráfico de Dona - Distribución por Región
# ============================================================

region_counts = df['REGION'].value_counts()

# Crear figura
fig, ax = plt.subplots(figsize=(12, 10))

# Colores
colores = [COLORES_REGIONES.get(r, '#999') for r in region_counts.index]

# Gráfico de dona
wedges, texts, autotexts = ax.pie(
    region_counts.values,
    labels=None,
    colors=colores,
    autopct='%1.1f%%',
    pctdistance=0.78,
    startangle=90,
    wedgeprops=dict(width=0.45, edgecolor='white', linewidth=2)
)

# Estilo de los porcentajes
for autotext in autotexts:
    autotext.set_fontsize(12)
    autotext.set_fontweight('bold')
    autotext.set_color('#333333')

# Etiquetas personalizadas con cantidad y porcentaje
legend_labels = [f'{r}: {c} municipios ({c/len(df)*100:.1f}%)'
                 for r, c in zip(region_counts.index, region_counts.values)]
ax.legend(wedges, legend_labels, loc='center left', bbox_to_anchor=(0.92, 0.5),
          fontsize=11, frameon=True, fancybox=True, shadow=True)

# Texto central
ax.text(0, 0, f'{len(df)}\nMunicipios\nTotales', ha='center', va='center',
        fontsize=18, fontweight='bold', color='#333333')

ax.set_title('Distribución Porcentual de Municipios por Región\nProyecto Nacional de Fibra Óptica',
             fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('grafico_03_dona_regiones.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: grafico_03_dona_regiones.png")

In [ ]:
# ============================================================
# GRÁFICO 4: Treemap de Municipios por Departamento y Región
# ============================================================

try:
    import squarify
    usar_squarify = True
except ImportError:
    usar_squarify = False
    print("⚠️ squarify no disponible, usando barras apiladas")

if usar_squarify:
    # Preparar datos
    dept_region = df.groupby(['REGION', 'DEPARTAME_NOMBRE']).size().reset_index(name='count')
    dept_region = dept_region.sort_values('count', ascending=False)

    # Etiquetas
    labels = [f"{r}\n{d}\n({c})" for r, d, c in
              zip(dept_region['REGION'], dept_region['DEPARTAME_NOMBRE'], dept_region['count'])]
    sizes = dept_region['count'].values

    # Colores por región
    colors = [COLORES_REGIONES.get(r, '#999') for r in dept_region['REGION']]

    # Crear figura
    fig, ax = plt.subplots(figsize=(18, 12))

    squarify.plot(sizes=sizes, label=labels, color=colors, alpha=0.85,
                 edgecolor='white', linewidth=2, text_kwargs={'fontsize': 6, 'fontweight': 'bold'})

    ax.set_title('Treemap: Distribución de Municipios por Departamento y Región',
                 fontsize=16, fontweight='bold', pad=20)
    ax.axis('off')

    # Leyenda
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=c, label=r) for r, c in COLORES_REGIONES.items()]
    ax.legend(handles=legend_elements, loc='upper right', fontsize=10, title='Región')

    plt.tight_layout()
    plt.savefig('grafico_04_treemap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Gráfico guardado: grafico_04_treemap.png")

else:
    # Alternativa: barras apiladas horizontales por región
    dept_region = df.groupby(['REGION', 'DEPARTAME_NOMBRE']).size().unstack(fill_value=0).T
    dept_region['Total'] = dept_region.sum(axis=1)
    dept_region = dept_region.sort_values('Total', ascending=True).drop('Total', axis=1)

    fig, ax = plt.subplots(figsize=(16, 14))
    dept_region.plot(kind='barh', stacked=True, ax=ax,
                     color=[COLORES_REGIONES.get(c, '#999') for c in dept_region.columns])

    ax.set_xlabel('Número de Municipios')
    ax.set_title('Distribución de Municipios por Departamento y Región (Barras Apiladas)',
                 fontsize=14, fontweight='bold')
    ax.legend(title='Región', bbox_to_anchor=(1.02, 1), loc='upper left')

    plt.tight_layout()
    plt.savefig('grafico_04_barras_apiladas.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Gráfico guardado: grafico_04_barras_apiladas.png")

In [ ]:
# ============================================================
# GRÁFICO 5: Mapa de Calor - Departamentos por Región
# ============================================================

# Tabla cruzada: departamentos vs regiones
cross_tab = pd.crosstab(df['DEPARTAME_NOMBRE'], df['REGION'])

# Ordenar por total
cross_tab['TOTAL'] = cross_tab.sum(axis=1)
cross_tab = cross_tab.sort_values('TOTAL', ascending=False).drop('TOTAL', axis=1)

# Crear figura
fig, ax = plt.subplots(figsize=(14, 14))

# Mapa de calor
sns.heatmap(cross_tab, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5,
            linecolor='white', ax=ax, cbar_kws={'label': 'N° de Municipios', 'shrink': 0.8})

ax.set_xlabel('Región', fontsize=13, fontweight='bold')
ax.set_ylabel('Departamento', fontsize=13, fontweight='bold')
ax.set_title('Mapa de Calor: Número de Municipios por Departamento y Región',
             fontsize=16, fontweight='bold', pad=20)

plt.yticks(rotation=0, fontsize=9)
plt.xticks(rotation=30, ha='right', fontsize=10)

plt.tight_layout()
plt.savefig('grafico_05_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: grafico_05_heatmap.png")

---
# 📅 3. Análisis Temporal

Estudiamos la evolución temporal del despliegue de fibra óptica en los municipios
colombianos, identificando los periodos de mayor actividad de instalación.

In [ ]:
# ============================================================
# GRÁFICO 6: Línea de Tiempo de Despliegues por Año y Mes
# ============================================================

# Despliegues por mes
despliegues_mes = df.groupby(df['FECHA OPERACION'].dt.to_period('M')).size()
despliegues_mes.index = despliegues_mes.index.to_timestamp()

# Despliegues por año
despliegues_anio = df.groupby('AÑO_INICIO').size()

# Figura con dos subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 12), gridspec_kw={'height_ratios': [1, 1]})

# Subplot 1: Despliegues por mes
ax1.fill_between(despliegues_mes.index, despliegues_mes.values, alpha=0.3, color='#4ECDC4')
ax1.plot(despliegues_mes.index, despliegues_mes.values, color='#4ECDC4', linewidth=2, marker='o', markersize=4)

# Etiquetas en los picos
max_idx = despliegues_mes.values.argmax()
ax1.annotate(f'Pico: {despliegues_mes.values[max_idx]} municipios',
             xy=(despliegues_mes.index[max_idx], despliegues_mes.values[max_idx]),
             xytext=(20, 20), textcoords='offset points', fontsize=10,
             arrowprops=dict(arrowstyle='->', color='red'),
             bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.8))

ax1.set_xlabel('Fecha', fontsize=12, fontweight='bold')
ax1.set_ylabel('Nuevos Municipios', fontsize=12, fontweight='bold')
ax1.set_title('Despliegues de Fibra Óptica por Mes', fontsize=14, fontweight='bold')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Subplot 2: Despliegues por año
colores_anio = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']
bars = ax2.bar(despliegues_anio.index.astype(int), despliegues_anio.values,
               color=colores_anio[:len(despliegues_anio)], edgecolor='white', linewidth=1.5, width=0.6)

# Etiquetas
for bar, valor in zip(bars, despliegues_anio.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             str(valor), ha='center', va='bottom', fontsize=12, fontweight='bold')

ax2.set_xlabel('Año de Inicio de Operación', fontsize=12, fontweight='bold')
ax2.set_ylabel('Número de Municipios', fontsize=12, fontweight='bold')
ax2.set_title('Despliegues de Fibra Óptica por Año', fontsize=14, fontweight='bold')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Fechas clave
ax2.axvline(x=2013, color='red', linestyle='--', alpha=0.5, label='Mayor despliegue (2013)')
ax2.legend(fontsize=10)

plt.tight_layout()
plt.savefig('grafico_06_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: grafico_06_timeline.png")

In [ ]:
# ============================================================
# GRÁFICO 7: Despliegue Acumulado en el Tiempo
# ============================================================

# Despliegue acumulado por mes
despliegues_mes = df.groupby(df['FECHA OPERACION'].dt.to_period('M')).size()
despliegues_mes.index = despliegues_mes.index.to_timestamp()
despliegues_acumulado = despliegues_mes.cumsum()

# Despliegue acumulado por año
despliegues_anio = df.groupby('AÑO_INICIO').size().sort_index()
despliegues_anio_acum = despliegues_anio.cumsum()

# Figura
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 12), gridspec_kw={'height_ratios': [1, 1]})

# Subplot 1: Acumulado por mes
ax1.fill_between(despliegues_acumulado.index, despliegues_acumulado.values,
                  alpha=0.4, color='#45B7D1', step='mid')
ax1.step(despliegues_acumulado.index, despliegues_acumulado.values,
         where='mid', color='#2C3E50', linewidth=2)

# Hitos
hito_100 = despliegues_acumulado[despliegues_acumulado >= 100].index[0]
hito_500 = despliegues_acumulado[despliegues_acumulado >= 500].index[0]

ax1.axhline(y=100, color='orange', linestyle='--', alpha=0.7)
ax1.axhline(y=500, color='red', linestyle='--', alpha=0.7)
ax1.text(despliegues_acumulado.index[-1], 100, f'  100 municipios ({hito_100.strftime("%b %Y")})',
         va='bottom', fontsize=9, color='orange')
ax1.text(despliegues_acumulado.index[-1], 500, f'  500 municipios ({hito_500.strftime("%b %Y")})',
         va='bottom', fontsize=9, color='red')

ax1.set_xlabel('Fecha', fontsize=12, fontweight='bold')
ax1.set_ylabel('Municipios Acumulados', fontsize=12, fontweight='bold')
ax1.set_title('Evolución Acumulada de Municipios con Fibra Óptica (por Mes)',
              fontsize=14, fontweight='bold')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Subplot 2: Acumulado por año
bars = ax2.bar(despliegues_anio_acum.index.astype(int), despliegues_anio_acum.values,
               color='#45B7D1', edgecolor='white', linewidth=1.5, width=0.6, alpha=0.8)

for bar, valor in zip(bars, despliegues_anio_acum.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             str(valor), ha='center', va='bottom', fontsize=12, fontweight='bold')

# Línea de tendencia
ax2.plot(despliegues_anio_acum.index.astype(int), despliegues_anio_acum.values,
         'ko-', linewidth=2, markersize=8, zorder=5)

ax2.set_xlabel('Año', fontsize=12, fontweight='bold')
ax2.set_ylabel('Municipios Acumulados', fontsize=12, fontweight='bold')
ax2.set_title('Evolución Acumulada de Municipios con Fibra Óptica (por Año)',
              fontsize=14, fontweight='bold')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('grafico_07_acumulado.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: grafico_07_acumulado.png")

In [ ]:
# ============================================================
# GRÁFICO 8: Despliegues por Año y Región (Barras Apiladas)
# ============================================================

# Tabla cruzada: año vs región
anio_region = pd.crosstab(df['AÑO_INICIO'], df['REGION'])

# Crear figura
fig, ax = plt.subplots(figsize=(14, 8))

# Barras apiladas
colores_stack = [COLORES_REGIONES.get(r, '#999') for r in anio_region.columns]
anio_region.plot(kind='bar', stacked=True, ax=ax, color=colores_stack,
                 edgecolor='white', linewidth=1, width=0.65)

# Etiquetas de total en la parte superior
for i, (anio, row) in enumerate(anio_region.iterrows()):
    total = row.sum()
    ax.text(i, total + 5, str(total), ha='center', va='bottom',
            fontsize=12, fontweight='bold')

ax.set_xlabel('Año de Inicio de Operación', fontsize=13, fontweight='bold')
ax.set_ylabel('Número de Municipios', fontsize=13, fontweight='bold')
ax.set_title('Distribución de Despliegues de Fibra Óptica por Año y Región',
             fontsize=16, fontweight='bold', pad=20)

# Configurar eje X
ax.set_xticklabels([int(x) for x in anio_region.index], rotation=0)
ax.legend(title='Región', bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=10, title_fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('grafico_08_barras_apiladas_anio_region.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: grafico_08_barras_apiladas_anio_region.png")

# Mostrar tabla de detalle
print("\n📋 Tabla de despliegues por año y región:")
display(anio_region)

---
# 💰 4. Análisis de Inversión

Profundizamos en el análisis de las inversiones del proyecto, comparando los recursos
del FONTIC con las contrapartidas departamentales y municipales, y analizando las
diferencias entre regiones y departamentos.

In [ ]:
# ============================================================
# GRÁFICO 9: Diagrama de Caja - Inversión Total por Región
# ============================================================

# Ordenar regiones por mediana
orden_region = df.groupby('REGION')['INVERSION TOTAL'].median().sort_values(ascending=False).index

# Crear figura
fig, ax = plt.subplots(figsize=(14, 8))

# Box plot
bp = sns.boxplot(data=df, x='REGION', y='INVERSION_TOTAL_BILLONES', order=orden_region,
                 palette=COLORES_REGIONES, ax=ax, width=0.5, fliersize=3,
                 boxprops=dict(alpha=0.7))

# Strip plot overlay
sns.stripplot(data=df, x='REGION', y='INVERSION_TOTAL_BILLONES', order=orden_region,
              color='#333333', alpha=0.15, size=3, jitter=True, ax=ax)

# Mediana labels
mediana_region = df.groupby('REGION')['INVERSION_TOTAL_BILLONES'].median()
for i, region in enumerate(orden_region):
    mediana = mediana_region[region]
    ax.text(i, ax.get_ylim()[1] * 0.98, f'${mediana:.2f}B',
            ha='center', va='top', fontsize=9, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

ax.set_xlabel('Región', fontsize=13, fontweight='bold')
ax.set_ylabel('Inversión Total (Billones COP)', fontsize=13, fontweight='bold')
ax.set_title('Distribución de la Inversión Total por Región',
             fontsize=16, fontweight='bold', pad=20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xticks(rotation=15, ha='right')

plt.tight_layout()
plt.savefig('grafico_09_boxplot_inversion_region.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: grafico_09_boxplot_inversion_region.png")

In [ ]:
# ============================================================
# GRÁFICO 10: Barras Agrupadas - FONTIC vs Contrapartida por Región
# ============================================================

# Promedio de inversión por región
inversion_region = df.groupby('REGION').agg({
    'INVERSION_FONTIC_BILLONES': 'mean',
    'INVERSION_CONTRAPARTIDA_BILLONES': 'mean',
    'INVERSION_TOTAL_BILLONES': 'mean' # Agregamos la inversión total a la agregación
}).sort_values('INVERSION_TOTAL_BILLONES', ascending=False) # Ahora podemos ordenar directamente por esta columna

# Crear figura
fig, ax = plt.subplots(figsize=(14, 8))

x = np.arange(len(inversion_region))
ancho = 0.35

# Barras agrupadas
bars1 = ax.bar(x - ancho/2, inversion_region['INVERSION_FONTIC_BILLONES'], ancho,
               label='FONTIC', color='#3498DB', edgecolor='white', linewidth=1)
bars2 = ax.bar(x + ancho/2, inversion_region['INVERSION_CONTRAPARTIDA_BILLONES'], ancho,
               label='Contrapartida', color='#E74C3C', edgecolor='white', linewidth=1)

# Etiquetas de valor
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'${bar.get_height():.2f}B', ha='center', va='bottom', fontsize=9, fontweight='bold')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'${bar.get_height():.2f}B', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xlabel('Región', fontsize=13, fontweight='bold')
ax.set_ylabel('Inversión Promedio (Billones COP)', fontsize=13, fontweight='bold')
ax.set_title('Comparación de Inversión Promedio: FONTIC vs Contrapartida por Región',
             fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels(inversion_region.index, rotation=15, ha='right')
ax.legend(fontsize=12, loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('grafico_10_fontic_vs_contrapartida.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: grafico_10_fontic_vs_contrapartida.png")

# Tabla resumen
print("\n📋 Resumen de inversión promedio por región:")
inversion_region.columns = ['FONTIC (Billones)', 'Contrapartida (Billones)', 'Total (Billones)'] # Actualizamos los nombres de las columnas
inversion_region['% FONTIC'] = (inversion_region['FONTIC (Billones)'] / inversion_region['Total (Billones)'] * 100).round(1)
inversion_region['% Contrapartida'] = (inversion_region['Contrapartida (Billones)'] / inversion_region['Total (Billones)'] * 100).round(1)
display(inversion_region)


In [ ]:
# ============================================================
# GRÁFICO 11: Composición de Inversión (Pie Chart)
# ============================================================

# Calcular totales
total_fontic = df['INVERSION FONTIC'].sum()
total_contra = df['INVERSION CONTRAPARTIDA'].sum()
total_inversion = total_fontic + total_contra
pct_fontic = total_fontic / total_inversion * 100
pct_contra = total_contra / total_inversion * 100

# Crear figura con dos subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

# Subplot 1: Composición global
sizes = [total_fontic, total_contra]
labels = [f'FONTIC\n${total_fontic/1e12:.2f}T COP\n({pct_fontic:.1f}%)',
          f'Contrapartida\n${total_contra/1e12:.2f}T COP\n({pct_contra:.1f}%)']
colors = ['#3498DB', '#E74C3C']
explode = (0.05, 0.05)

wedges, texts = ax1.pie(sizes, labels=labels, colors=colors, explode=explode,
                        startangle=90, textprops={'fontsize': 11, 'fontweight': 'bold'})

ax1.set_title('Composición Total de Inversión\nProyecto Fibra Óptica',
              fontsize=14, fontweight='bold')

# Subplot 2: Composición por región
comp_region = df.groupby('REGION').agg({
    'INVERSION FONTIC': 'sum',
    'INVERSION CONTRAPARTIDA': 'sum'
})
comp_region['TOTAL'] = comp_region.sum(axis=1)
comp_region = comp_region.sort_values('TOTAL', ascending=True)

# Barras apiladas horizontales
comp_region[['INVERSION FONTIC', 'INVERSION CONTRAPARTIDA']].plot(
    kind='barh', stacked=True, ax=ax2, color=['#3498DB', '#E74C3C'],
    edgecolor='white', linewidth=1)

ax2.set_xlabel('Inversión Total (COP)', fontsize=11, fontweight='bold')
ax2.set_title('Composición de Inversión por Región', fontsize=14, fontweight='bold')
ax2.legend(['FONTIC', 'Contrapartida'], fontsize=10)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Formatear eje X en billones
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x/1e12:.1f}T'))

plt.suptitle(f'Inversión Total del Proyecto: ${total_inversion/1e12:.2f} Billones COP',
             fontsize=16, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('grafico_11_composicion_inversion.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: grafico_11_composicion_inversion.png")

In [ ]:
# ============================================================
# GRÁFICO 12: Top 15 Departamentos por Inversión Total
# ============================================================

# Inversión total por departamento
inversion_dept = df.groupby('DEPARTAME_NOMBRE')['INVERSION TOTAL'].sum().sort_values(ascending=True)

# Tomar top 15
top_15_inv = inversion_dept.tail(15)

# Colores por región
dept_region_map = df.drop_duplicates('DEPARTAME_NOMBRE').set_index('DEPARTAME_NOMBRE')['REGION']
colores_dept_inv = [COLORES_REGIONES.get(dept_region_map.get(d, ''), '#999') for d in top_15_inv.index]

# Crear figura
fig, ax = plt.subplots(figsize=(14, 10))

bars = ax.barh(top_15_inv.index, top_15_inv.values / 1e12,
               color=colores_dept_inv, edgecolor='white', linewidth=1, height=0.7)

# Etiquetas
for bar, valor in zip(bars, top_15_inv.values):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'${valor/1e12:.2f}T COP', va='center', ha='left', fontsize=10, fontweight='bold')

ax.set_xlabel('Inversión Total (Billones COP)', fontsize=13, fontweight='bold')
ax.set_title('Top 15 Departamentos por Inversión Total en Fibra Óptica',
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlim(0, max(top_15_inv.values / 1e12) * 1.25)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='y', labelsize=10)

# Leyenda
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=r) for r, c in COLORES_REGIONES.items()]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9, title='Región', title_fontsize=10)

plt.tight_layout()
plt.savefig('grafico_12_top_departamentos_inversion.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: grafico_12_top_departamentos_inversion.png")

# Mostrar inversión total del proyecto
total = df['INVERSION TOTAL'].sum()
print(f"\n💰 Inversión total del proyecto: ${total/1e12:,.2f} billones COP")
print(f"💰 Inversión total del proyecto: ${total/1e15:,.2f} trillones COP")

In [ ]:
# ============================================================
# GRÁFICO 13: Dispersión - FONTIC vs Contrapartida por Región
# ============================================================

# Crear figura
fig, ax = plt.subplots(figsize=(14, 10))

# Scatter plot por región
for region in df['REGION'].unique():
    mask = df['REGION'] == region
    ax.scatter(df.loc[mask, 'INVERSION FONTIC'] / 1e9,
               df.loc[mask, 'INVERSION CONTRAPARTIDA'] / 1e9,
               c=COLORES_REGIONES.get(region, '#999'),
               label=region, alpha=0.6, s=60, edgecolors='white', linewidth=0.5)

# Línea de regresión
from numpy.polynomial import polynomial as P
x_data = df['INVERSION FONTIC'].values / 1e9
y_data = df['INVERSION CONTRAPARTIDA'].values / 1e9
coef = np.polyfit(x_data, y_data, 1)
x_line = np.linspace(x_data.min(), x_data.max(), 100)
y_line = np.polyval(coef, x_line)
ax.plot(x_line, y_line, 'k--', linewidth=2, alpha=0.7, label=f'Tendencia (r={np.corrcoef(x_data, y_data)[0,1]:.3f})')

ax.set_xlabel('Inversión FONTIC (Billones COP)', fontsize=13, fontweight='bold')
ax.set_ylabel('Inversión Contrapartida (Billones COP)', fontsize=13, fontweight='bold')
ax.set_title('Relación entre Inversión FONTIC y Contrapartida por Región',
             fontsize=16, fontweight='bold', pad=20)
ax.legend(fontsize=10, loc='upper left', frameon=True, fancybox=True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('grafico_13_scatter_fontic_contrapartida.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: grafico_13_scatter_fontic_contrapartida.png")

# Correlación
corr = np.corrcoef(df['INVERSION FONTIC'], df['INVERSION CONTRAPARTIDA'])[0, 1]
print(f"\n📈 Correlación FONTIC vs Contrapartida: {corr:.4f}")

In [ ]:
# ============================================================
# GRÁFICO 14: Gráfico de Burbujas - Inversión por Región
# ============================================================

# Resumen por región
resumen_region = df.groupby('REGION').agg({
    'MUNICIPIO_NOMBRE': 'count',
    'INVERSION_TOTAL_BILLONES': 'mean',
    'INVERSION TOTAL': 'sum'
}).rename(columns={'MUNICIPIO_NOMBRE': 'n_municipios',
                    'INVERSION_TOTAL_BILLONES': 'inv_promedio',
                    'INVERSION TOTAL': 'inv_total'})

# Crear figura
fig, ax = plt.subplots(figsize=(14, 10))

# Burbujas
for i, (region, row) in enumerate(resumen_region.iterrows()):
    ax.scatter(row['n_municipios'], row['inv_promedio'],
               s=row['inv_total'] / 5e10,  # Tamaño proporcional
               c=COLORES_REGIONES.get(region, '#999'),
               alpha=0.7, edgecolors='black', linewidth=1.5, zorder=5)
    ax.annotate(region, (row['n_municipios'], row['inv_promedio']),
                textcoords="offset points", xytext=(0, 20),
                ha='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Número de Municipios', fontsize=13, fontweight='bold')
ax.set_ylabel('Inversión Promedio por Municipio (Billones COP)', fontsize=13, fontweight='bold')
ax.set_title('Relación: N° de Municipios, Inversión Promedio e Inversión Total por Región\n'
             '(Tamaño de burbuja = Inversión Total)',
             fontsize=14, fontweight='bold', pad=20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Nota sobre tamaño
ax.text(0.02, 0.02, 'El tamaño de la burbuja representa la inversión total de la región',
        transform=ax.transAxes, fontsize=9, fontstyle='italic',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('grafico_14_burbujas_region.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: grafico_14_burbujas_region.png")

print("\n📋 Resumen por región:")
resumen_region_display = resumen_region.copy()
resumen_region_display['inv_total'] = resumen_region_display['inv_total'].apply(lambda x: f'${x/1e12:.2f}T')
display(resumen_region_display)

In [ ]:
# ============================================================
# GRÁFICO 15: Histograma + KDE de Inversión Total
# ============================================================

# Crear figura
fig, ax = plt.subplots(figsize=(14, 8))

# Histograma
n, bins, patches = ax.hist(df['INVERSION_TOTAL_BILLONES'], bins=30, alpha=0.6,
                           color='#4ECDC4', edgecolor='white', linewidth=1, density=True)

# KDE
sns.kdeplot(df['INVERSION_TOTAL_BILLONES'], color='#2C3E50', linewidth=2.5, ax=ax)

# Líneas de referencia
media = df['INVERSION_TOTAL_BILLONES'].mean()
mediana = df['INVERSION_TOTAL_BILLONES'].median()

ax.axvline(media, color='red', linestyle='--', linewidth=2, label=f'Media: ${media:.3f}B')
ax.axvline(mediana, color='blue', linestyle='-.', linewidth=2, label=f'Mediana: ${mediana:.3f}B')

ax.set_xlabel('Inversión Total (Billones COP)', fontsize=13, fontweight='bold')
ax.set_ylabel('Densidad', fontsize=13, fontweight='bold')
ax.set_title('Distribución de la Inversión Total por Municipio',
             fontsize=16, fontweight='bold', pad=20)
ax.legend(fontsize=12, loc='upper right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Anotación
ax.text(0.02, 0.95, f'Desv. Est.: ${df["INVERSION_TOTAL_BILLONES"].std():.3f}B\n'
        f'Mín: ${df["INVERSION_TOTAL_BILLONES"].min():.3f}B\n'
        f'Máx: ${df["INVERSION_TOTAL_BILLONES"].max():.3f}B',
        transform=ax.transAxes, fontsize=10, va='top',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

plt.tight_layout()
plt.savefig('grafico_15_histograma_inversion.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: grafico_15_histograma_inversion.png")

In [ ]:
# ============================================================
# GRÁFICO 16: Diagrama de Violín - Inversión por Región
# ============================================================

# Ordenar regiones por mediana
orden_region = df.groupby('REGION')['INVERSION_TOTAL_BILLONES'].median().sort_values(ascending=False).index

# Crear figura
fig, ax = plt.subplots(figsize=(14, 8))

# Violin plot
sns.violinplot(data=df, x='REGION', y='INVERSION_TOTAL_BILLONES', order=orden_region,
               palette=COLORES_REGIONES, ax=ax, inner='box', cut=0)

# Strip plot overlay
sns.stripplot(data=df, x='REGION', y='INVERSION_TOTAL_BILLONES', order=orden_region,
              color='#333333', alpha=0.1, size=3, jitter=True, ax=ax)

ax.set_xlabel('Región', fontsize=13, fontweight='bold')
ax.set_ylabel('Inversión Total (Billones COP)', fontsize=13, fontweight='bold')
ax.set_title('Distribución de Inversión Total por Región (Diagrama de Violín)',
             fontsize=16, fontweight='bold', pad=20)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.xticks(rotation=15, ha='right')

plt.tight_layout()
plt.savefig('grafico_16_violin_inversion.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: grafico_16_violin_inversion.png")

# Estadísticas por región
print("\n📋 Estadísticas de inversión por región:")
stats_region = df.groupby('REGION')['INVERSION_TOTAL_BILLONES'].agg(['mean', 'median', 'std', 'min', 'max', 'count'])
stats_region.columns = ['Media', 'Mediana', 'Desv. Est.', 'Mínimo', 'Máximo', 'N']
stats_region = stats_region.sort_values('Media', ascending=False)
display(stats_region.round(3))

---
# 🔗 5. Correlaciones y Relaciones

Analizamos las correlaciones entre las variables numéricas del dataset para
identificar relaciones significativas entre las diferentes métricas de inversión.

In [ ]:
# ============================================================
# GRÁFICO DE CORRELACIÓN: Matriz de Correlación
# ============================================================

# Seleccionar columnas numéricas
columnas_corr = ['INVERSION FONTIC', 'INVERSION CONTRAPARTIDA', 'INVERSION TOTAL',
                 'DURACION_AÑOS', 'AÑO_INICIO', 'PORCENTAJE_FONTIC', 'PORCENTAJE_CONTRAPARTIDA',
                 'INVERSION_TOTAL_BILLONES']

# Calcular correlación
corr_matrix = df[columnas_corr].corr()

# Crear figura
fig, ax = plt.subplots(figsize=(12, 10))

# Heatmap de correlación
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            mask=mask, square=True, linewidths=0.5, linecolor='white',
            ax=ax, cbar_kws={'label': 'Coeficiente de Correlación', 'shrink': 0.8},
            annot_kws={'fontsize': 10})

ax.set_title('Matriz de Correlación entre Variables Numéricas',
             fontsize=16, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('grafico_correlacion.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: grafico_correlacion.png")

# Interpretación de correlaciones clave
print("\n📋 Interpretación de correlaciones clave:")
print(f"   • INVERSION FONTIC vs INVERSION TOTAL: {corr_matrix.loc['INVERSION FONTIC', 'INVERSION TOTAL']:.3f}")
print(f"   • INVERSION CONTRAPARTIDA vs INVERSION TOTAL: {corr_matrix.loc['INVERSION CONTRAPARTIDA', 'INVERSION TOTAL']:.3f}")
print(f"   • AÑO_INICIO vs INVERSION TOTAL: {corr_matrix.loc['AÑO_INICIO', 'INVERSION TOTAL']:.3f}")
print(f"   • INVERSION FONTIC vs INVERSION CONTRAPARTIDA: {corr_matrix.loc['INVERSION FONTIC', 'INVERSION CONTRAPARTIDA']:.3f}")

In [ ]:
# ============================================================
# GRÁFICO 17: Dispersión a Nivel Departamental
# ============================================================

# Resumen por departamento
resumen_dept = df.groupby(['DEPARTAME_NOMBRE', 'REGION']).agg({
    'MUNICIPIO_NOMBRE': 'count',
    'INVERSION_TOTAL_BILLONES': 'mean',
    'INVERSION TOTAL': 'sum'
}).reset_index()
resumen_dept.columns = ['Departamento', 'Región', 'N_Municipios', 'Inv_Promedio', 'Inv_Total']

# Crear figura
fig, ax = plt.subplots(figsize=(14, 10))

# Scatter por región
for region in resumen_dept['Región'].unique():
    mask = resumen_dept['Región'] == region
    ax.scatter(resumen_dept.loc[mask, 'N_Municipios'],
               resumen_dept.loc[mask, 'Inv_Promedio'],
               s=resumen_dept.loc[mask, 'Inv_Total'] / 3e10,
               c=COLORES_REGIONES.get(region, '#999'),
               label=region, alpha=0.7, edgecolors='black', linewidth=0.5)

# Etiquetar departamentos destacados
destacados = resumen_dept.nlargest(5, 'Inv_Total')
for _, row in destacados.iterrows():
    ax.annotate(row['Departamento'],
                (row['N_Municipios'], row['Inv_Promedio']),
                textcoords="offset points", xytext=(10, 5),
                fontsize=8, fontweight='bold', color='darkred')

ax.set_xlabel('Número de Municipios en el Departamento', fontsize=13, fontweight='bold')
ax.set_ylabel('Inversión Promedio por Municipio (Billones COP)', fontsize=13, fontweight='bold')
ax.set_title('Relación entre N° de Municipios e Inversión Promedio por Departamento\n'
             '(Tamaño = Inversión Total del Departamento)',
             fontsize=14, fontweight='bold', pad=20)
ax.legend(fontsize=10, loc='upper right', title='Región')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('grafico_17_dispersion_departamental.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Gráfico guardado: grafico_17_dispersion_departamental.png")

---
# 📋 6. Resumen de Hallazgos

### 🔑 Hallazgos Principales

#### 📍 Distribución Geográfica
- El proyecto beneficia **788 municipios** distribuidos en **6 regiones** y **27 departamentos** de Colombia.
- La **Región Centro Oriente** concentra la mayor cantidad de municipios beneficiados, liderada por Boyacá (117 municipios) y Cundinamarca (89 municipios).
- **Boyacá** es el departamento con más municipios cubiertos (117), seguido por Cundinamarca (89) y Santander (78).
- La **Región Llano** tiene la menor cobertura en términos de número de municipios.

#### 📅 Evolución Temporal
- La gran mayoría de los despliegues se realizaron en **2013 (452 municipios, 57.4%)** y **2014 (317 municipios, 40.2%)**.
- Solo **15 municipios** iniciaron operaciones en 2015, **2 en 2016** y **2 en 2023**.
- El pico de despliegues mensuales se alcanzó en los primeros meses de 2013.
- Todos los contratos tienen una duración de operación de aproximadamente **15 años**.

#### 💰 Inversión
- La inversión **FONTIC es constante** para todos los municipios ($552.241.234 COP).
- La inversión de **contrapartida varía significativamente** entre departamentos, desde ~$487M hasta ~$1.007M.
- La inversión total por municipio oscila entre ~$1.039M y ~$1.559M COP.
- El **FONTIC representa aproximadamente el 43%** de la inversión total, mientras que la **contrapartida representa el 57%**.
- Los departamentos de la Región Llano (Vichada, Casanare) y Pacífico (Chocó) tienen las inversiones de contrapartida más altas.

#### 🔗 Correlaciones
- Existe una **correlación negativa moderada** entre el año de inicio y la inversión total, sugiriendo que los proyectos más recientes tienden a tener menor inversión.
- La **inversión FONTIC es constante**, por lo que las variaciones en inversión total se deben exclusivamente a las contrapartidas.
- Los departamentos con más municipios tienden a tener contrapartidas más uniformes.

#### ⚡ Observaciones Especiales
- Todos los municipios están en estado **"En Operación"**, indicando una ejecución exitosa del proyecto.
- Los **2 municipios de 2023** representan una posible reactivación o ampliación reciente del proyecto.
- La información fue cargada al sistema el **31 de octubre de 2023**.

---
# 🎯 7. Lineamientos para Exploración Adicional

### 📊 Sugerencias para Análisis Futuros

#### 1. 🗺️ Conexión con Datos Demográficos (DANE)
- Cruzar los datos de fibra óptica con **población por municipio** del DANE para calcular cobertura per cápita.
- Analizar la relación entre **densidad poblacional** y el nivel de inversión.
- Evaluar si los municipios más pequeños reciben proporcionalmente más inversión.
- Incorporar datos de **pobreza multidimensional** (índice SISBÉN) para analizar equidad.

#### 2. 🌐 Mapeo Geográfico con Folium/Geopandas
- Crear **mapas interactivos** que muestren la ubicación exacta de cada municipio con fibra óptica.
- Utilizar **GeoJSON** de los límites municipales para crear coropletas.
- Superponer el mapa con datos de **cobertura de internet** del MinTIC.
- Analizar la **conectividad entre municipios** (red de fibra óptica).

#### 3. 📡 Comparación con Datos de Cobertura de Internet
- Conectar con los datos del **Observatorio MinTIC** sobre penetración de internet.
- Comparar los municipios con fibra óptica vs. los que aún no tienen.
- Analizar la **evolución de la velocidad de internet** antes y después del despliegue.
- Estudiar el impacto en la **brecha digital** urbano-rural.

#### 4. 📈 Análisis de Series Temporales
- Modelar la **tasa de adopción de internet** antes y después del despliegue.
- Predecir cuándo se alcanzaría la **cobertura universal** con la tasa actual.
- Analizar el impacto estacional en la **demanda de conectividad**.
- Estudiar la correlación entre fibra óptica y el **crecimiento económico local**.

#### 5. 💡 Análisis de Retorno de Inversión (ROI)
- Calcular el **costo por habitante atendido** por municipio.
- Comparar con el **PIB per cápita** departamental.
- Estimar el impacto en **productividad y emprendimiento digital**.
- Analizar la relación costo-beneficio comparando regiones.

#### 6. 🔬 Análisis Avanzado con Machine Learning
- Aplicar **clustering** para identificar grupos de municipios con características similares.
- Utilizar **regresión** para modelar la inversión necesaria según características municipales.
- Crear un **modelo predictivo** para priorizar municipios futuros.
- Realizar **análisis de componentes principales** (PCA) para reducción dimensional.

#### 7. 📊 Dashboard Interactivo
- Desarrollar un **dashboard con Plotly Dash o Streamlit** para visualización interactiva.
- Incluir filtros por región, departamento, año y rango de inversión.
- Crear **KPIs automáticos** de seguimiento del proyecto.
- Exportar resultados a **formato web** para compartir con interesados.

#### 8. 📋 Datos Complementarios Sugeridos
- Base de datos de **municipios del DANE** (georreferenciación)
- Datos del **SISBEN** para análisis socioeconómico
- Estadísticas de **MinTIC** sobre conectividad
- Datos del **Banco Mundial** sobre desarrollo digital
- Información del **CNMC** sobre calidad del servicio

---
# 🗺️ 8. Visualizaciones Interactivas con Plotly

Creamos visualizaciones interactivas que permiten explorar los datos con mayor profundidad.

In [ ]:
# ============================================================
# GRÁFICO INTERACTIVO 1: Barras de Inversión por Región
# ============================================================

# Resumen por región
resumen_region = df.groupby('REGION').agg(
    Municipios=('MUNICIPIO_NOMBRE', 'count'),
    Inv_Total=('INVERSION TOTAL', 'sum'),
    Inv_Promedio=('INVERSION TOTAL', 'mean'),
    Inv_Fontic=('INVERSION FONTIC', 'sum'),
    Inv_Contrapartida=('INVERSION CONTRAPARTIDA', 'sum')
).reset_index()

fig = px.bar(resumen_region.sort_values('Inv_Total', ascending=True),
             x='Inv_Total', y='REGION', orientation='h',
             color='REGION',
             hover_data=['Municipios', 'Inv_Promedio', 'Inv_Fontic', 'Inv_Contrapartida'],
             title='Inversión Total por Región (Interactivo)',
             labels={'Inv_Total': 'Inversión Total (COP)', 'REGION': 'Región'},
             color_discrete_map=COLORES_REGIONES)

fig.update_layout(
    height=600, width=1000,
    hoverlabel=dict(bgcolor='white', font_size=12),
    xaxis_title='Inversión Total (COP)',
    yaxis_title='',
    title_font_size=16
)

fig.show()

In [ ]:
# ============================================================
# GRÁFICO INTERACTIVO 2: Dispersión FONTIC vs Contrapartida
# ============================================================

fig = px.scatter(df, x='INVERSION FONTIC', y='INVERSION CONTRAPARTIDA',
                 color='REGION', hover_name='MUNICIPIO_NOMBRE',
                 hover_data=['DEPARTAME_NOMBRE', 'INVERSION TOTAL', 'AÑO_INICIO'],
                 title='Relación FONTIC vs Contrapartida por Municipio (Interactivo)',
                 labels={'INVERSION FONTIC': 'Inversión FONTIC (COP)',
                         'INVERSION CONTRAPARTIDA': 'Inversión Contrapartida (COP)'},
                 color_discrete_map=COLORES_REGIONES,
                 size='INVERSION TOTAL', size_max=15, opacity=0.7)

fig.update_layout(
    height=700, width=1100,
    title_font_size=16
)

fig.show()

In [ ]:
# ============================================================
# GRÁFICO INTERACTIVO 3: Línea Temporal de Despliegues
# ============================================================

despliegues_mes_df = df.groupby(df['FECHA OPERACION'].dt.to_period('M')).size().reset_index()
despliegues_mes_df.columns = ['Fecha', 'Municipios']
despliegues_mes_df['Fecha'] = despliegues_mes_df['Fecha'].astype(str)

fig = px.line(despliegues_mes_df, x='Fecha', y='Municipios',
              title='Evolución Temporal de Despliegues de Fibra Óptica (Interactivo)',
              labels={'Fecha': 'Fecha', 'Municipios': 'Nuevos Municipios'})

fig.update_traces(line_color='#4ECDC4', line_width=3)
fig.add_scatter(x=despliegues_mes_df['Fecha'], y=despliegues_mes_df['Municipios'],
                mode='markers+lines', name='Despliegues',
                marker=dict(size=8, color='#E74C3C'))

fig.update_layout(
    height=500, width=1100,
    title_font_size=16,
    xaxis_title='Fecha',
    yaxis_title='Nuevos Municipios'
)

fig.show()

In [ ]:
# ============================================================
# GRÁFICO INTERACTIVO 4: Treemap de Departamentos
# ============================================================

inv_dept = df.groupby(['REGION', 'DEPARTAME_NOMBRE']).agg(
    Municipios=('MUNICIPIO_NOMBRE', 'count'),
    Inv_Total=('INVERSION TOTAL', 'sum')
).reset_index()

fig = px.treemap(inv_dept, path=[px.Constant('Colombia'), 'REGION', 'DEPARTAME_NOMBRE'],
                 values='Inv_Total', color='REGION',
                 color_discrete_map=COLORES_REGIONES,
                 title='Treemap de Inversión por Región y Departamento (Interactivo)',
                 hover_data=['Municipios'])

fig.update_layout(
    height=700, width=1100,
    title_font_size=16
)

fig.show()

---
# ✅ Conclusión

## 📊 Resumen del Análisis

Este notebook ha realizado un análisis exploratorio exhaustivo del **Proyecto Nacional de Fibra Óptica de Colombia**,
cubriendo los siguientes aspectos:

| Sección | Gráficos | Hallazgos Clave |
|---------|----------|-----------------|
| **Distribución Geográfica** | 5 gráficos | Concentración en Centro Oriente; Boyacá lidera |
| **Análisis Temporal** | 3 gráficos | 97.6% de despliegues en 2013-2014 |
| **Análisis de Inversión** | 8 gráficos | FONTIC constante; contrapartida variable |
| **Correlaciones** | 2 gráficos | Relación inversa año-inversión |
| **Interactivos** | 4 gráficos | Exploración dinámica de datos |

### 📁 Archivos Generados
- Notebook: `Analisis_Fibra_Optica_Colombia.ipynb`
- Gráficos PNG (16 visualizaciones estáticas)

---

*Análisis realizado con Python, Pandas, Matplotlib, Seaborn y Plotly*  
*Datos fuente: MinTIC - Proyecto Nacional de Fibra Óptica*  
*Fecha de análisis: Marzo 2025*